# w9_pi2ccec.ipynb — DEDICATED run: pi2ccec@g2048 (projected per-source I/C heads)

Single-job notebook (user decree: this experiment is important — do NOT run it
inside w9_all's queue). Same protocol/infra as w9_all: repo force-sync, corpus
in RAM, full pool staged to fast local storage, shared claims on the volume so
concurrent w9_all pods will not double-run it. Results land in the SAME
`/workspace/w9_out` (`ft4var_w9_wcle_pi2ccec_icetf_g2048_fp_best.json`).
A/B partner: `i2ccec@g2048` (m4 0.829, tag 0.731). AUTO-STOPS the pod when done.


In [ ]:
# constants
import os, subprocess

REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_DIR = "/workspace/w9_out"          # same out dir as the campaign

ARM, CAP = "wcle_pi2ccec_icetf", 2048
EPOCHS, CKPT_EVERY, CKPT_SEEDS, TOPUP_SEEDS = 1000, 50, 3, 10
NM = f"w9_{ARM}_g{CAP}"                # claim/log/result stem (worker adds _fp)
os.makedirs(OUT_DIR, exist_ok=True)
print("job:", NM)


In [ ]:
# FORCE-sync repo to origin/main.
import os, importlib.util
if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}
%cd {REPO}
!git remote set-url origin {URL}
!git fetch origin main
!git reset --hard origin/main
!git rev-parse --short HEAD
for pkg in ("sklearn", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        !pip -q install scikit-learn scipy
        break


In [ ]:
# Stage the corpus into RAM (same file set as w9_all, minus llm views).
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "sp_raw_views.npz",
            "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
print("corpus in RAM:", DATA_DIR)


In [ ]:
# Full pool: must already be READY on the volume (built by the campaign);
# stage it onto fast local storage (h5_staging.parallel_copy, volume fallback).
import os, sys
from pathlib import Path
if REPO not in sys.path:
    sys.path.insert(0, REPO)
from Pod.h5_staging import parallel_copy

ready = Path(DATA_SRC) / "full_pool_READY"
assert ready.exists(), "full pool not READY on the volume -- run w9_all cell 4 once"
src_v = Path(DATA_SRC) / "full_pool_fp16.npy"
src_m = Path(DATA_SRC) / "full_pool_meta.npz"
need = src_v.stat().st_size + (5 << 30)

def _free(p):
    st = os.statvfs(p)
    return st.f_bavail * st.f_frsize

dest_dir = None
for cand in ("/dev/shm", "/root/data", "/root"):
    Path(cand).mkdir(parents=True, exist_ok=True)
    if _free(cand) > need:
        dest_dir = Path(cand)
        break
if dest_dir is None:
    print("WARNING: no local space -- the worker will mmap the NETWORK VOLUME copy.")
    FULL_POOL_PATH = str(src_v)
else:
    dst_v = dest_dir / "full_pool_fp16.npy"
    if dst_v.exists() and dst_v.stat().st_size == src_v.stat().st_size:
        print("local full pool already staged:", dst_v)
    else:
        import time
        t0 = time.time()
        tmp = dst_v.with_name(dst_v.name + ".copying")
        print(f"staging {src_v.stat().st_size/2**30:.0f} GiB -> {dst_v} ...", flush=True)
        parallel_copy(src_v, tmp, workers=8)
        os.replace(tmp, dst_v)
        print(f"staged in {(time.time()-t0)/60:.1f} min", flush=True)
    import shutil
    shutil.copyfile(src_m, dest_dir / "full_pool_meta.npz")
    FULL_POOL_PATH = str(dst_v)
print("FULL_POOL_PATH =", FULL_POOL_PATH)


In [ ]:
# Claim + run THE job (foreground; heartbeat prints the log tail every 3 min).
import os, socket, subprocess, threading, time
from pathlib import Path

HOST = socket.gethostname() + ":" + os.environ.get("RUNPOD_POD_ID", "?")
CLAIM_STALE_H = 12
claim_dir = Path(OUT_DIR) / "claims"
claim_dir.mkdir(parents=True, exist_ok=True)
cl = claim_dir / f"{NM}.claim"
try:
    fd = os.open(cl, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
    os.write(fd, f"{HOST} {time.time():.0f}".encode()); os.close(fd)
    claimed = True
except FileExistsError:
    owner, ts = cl.read_text().split()
    claimed = (owner == HOST or time.time() - float(ts) > CLAIM_STALE_H * 3600)
    if claimed:
        cl.write_text(f"{HOST} {time.time():.0f}")
    else:
        print(f"claimed by {owner} -- NOT running here.")

done = Path(OUT_DIR) / f"ft4var_{NM}_fp_best.json"
if done.exists():
    print("already DONE:", done)
elif claimed:
    log = Path(OUT_DIR) / "logs" / f"{ARM}_g{CAP}.log"
    log.parent.mkdir(exist_ok=True)
    stop_evt = threading.Event()
    def _beat():
        while not stop_evt.wait(180):
            try:
                with open(log, "rb") as fh:
                    fh.seek(max(0, log.stat().st_size - 400))
                    tail = fh.read().decode(errors="ignore").strip().splitlines()
                if tail:
                    print(f"[beat] {tail[-1]}", flush=True)
            except Exception:
                pass
    threading.Thread(target=_beat, daemon=True).start()
    t0 = time.time()
    with open(log, "w") as fh:
        p = subprocess.run(
            ["python", "-u", os.path.join(REPO, "Pod/w9_a100_worker.py"),
             "--data-dir", DATA_DIR, "--out-dir", OUT_DIR, "--repo", REPO,
             "--arm", ARM, "--anchor-cap", str(CAP),
             "--epochs", str(EPOCHS), "--ckpt-every", str(CKPT_EVERY),
             "--ckpt-seeds", str(CKPT_SEEDS), "--topup-seeds", str(TOPUP_SEEDS),
             "--full-pool", "--full-pool-path", FULL_POOL_PATH],
            stdout=fh, stderr=subprocess.STDOUT,
            env=dict(os.environ, CUDA_VISIBLE_DEVICES="0"))
    stop_evt.set()
    print(("ok" if p.returncode == 0 else "FAIL")
          + f" [{(time.time()-t0)/60:.1f} min] -> {log}")


In [ ]:
# A/B readout vs i2ccec@g2048 (both from the shared OUT_DIR).
import json
import numpy as np
from pathlib import Path
VORD = ["neutral", "noname", "positive", "negative"]
for stem, lab in [(f"ft4var_{NM}_fp", "pi2ccec_g2048 (THIS)"),
                  ("ft4var_w9_wcle_i2ccec_icetf_g2048_fp", "i2ccec_g2048 (A/B)")]:
    p = Path(OUT_DIR) / f"{stem}_best.json"
    if not p.exists():
        print(f"{lab}: (missing)"); continue
    d = json.loads(p.read_text())
    runs = d["per_seed"]
    r = {v: np.mean([x[v]["h1"] for x in runs]) for v in VORD}
    print(f"{lab}: ep{d.get('best_ep')} "
          + " ".join(f"{v}:{r[v]:.3f}" for v in VORD)
          + f" m4:{np.mean(list(r.values())):.3f}"
          + f" tag:{np.mean([x['noname']['tag'] for x in runs]):.3f}")


In [ ]:
# AUTO-STOP the pod (results are on the network volume).
AUTO_STOP = True
if AUTO_STOP:
    import sys
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from VICReg_review import pod_selfstop
    pod_id, api_key, ctl = pod_selfstop.preflight("")
    pod_selfstop.stop_pod(pod_id, api_key, ctl)
else:
    print("AUTO_STOP disabled -- stop the pod yourself.")
